# NLP-Based BFRB Detection Model

This notebook builds and trains an NLP-based model for Body-Focused Repetitive Behavior (BFRB) detection.
It processes sensor data features as sequential text-like representations and applies NLP techniques for classification.

## 1. Imports and Configuration

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Paths
DATA_PATH = '../data/processed/cmi_sensor_data/train_mlp_scaled.csv'
VAL_DATA_PATH = '../data/processed/cmi_sensor_data/val_mlp_scaled.csv'
METADATA_PATH = '../models_artifacts/metadata/class_mapping.json'
OUTPUT_DIR = '../models_artifacts/outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load Preprocessed Data

In [ ]:
train_df = pd.read_csv(DATA_PATH)
val_df = pd.read_csv(VAL_DATA_PATH)

print(f'Train shape: {train_df.shape}')
print(f'Val shape:   {val_df.shape}')
train_df.head()

## 3. Feature Extraction and Label Encoding

In [ ]:
# Load class mapping
with open(METADATA_PATH, 'r') as f:
    class_mapping = json.load(f)

label_col = 'label'

X_train = train_df.drop(columns=[label_col]).values
y_train = train_df[label_col].values

X_val = val_df.drop(columns=[label_col]).values
y_val = val_df[label_col].values

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

print(f'Classes: {le.classes_}')
print(f'X_train shape: {X_train.shape}, y_train shape: {y_train_enc.shape}')

## 4. NLP-Inspired Feature Representation

Sensor time-series windows are tokenized into discrete bins (analogous to words) using quantile-based binning. Each sequence of bins forms a "sentence" that is passed to a TF-IDF-style feature extractor.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

N_BINS = 50

def tokenize_sensor_row(row, n_bins=N_BINS):
    """Convert a numeric feature vector into a sequence of bin tokens."""
    bins = np.linspace(row.min(), row.max() + 1e-9, n_bins + 1)
    token_ids = np.digitize(row, bins) - 1
    return ' '.join([f't{i}b{b}' for i, b in enumerate(token_ids)])

train_docs = [tokenize_sensor_row(row) for row in X_train]
val_docs   = [tokenize_sensor_row(row) for row in X_val]

print(f'Sample token sequence (first 80 chars): {train_docs[0][:80]}...')

## 5. TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(train_docs)
X_val_tfidf   = vectorizer.transform(val_docs)

print(f'TF-IDF train matrix: {X_train_tfidf.shape}')
print(f'TF-IDF val matrix:   {X_val_tfidf.shape}')

## 6. Model Training — Logistic Regression (NLP Baseline)

In [ ]:
from sklearn.linear_model import LogisticRegression

nlp_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    random_state=RANDOM_SEED,
    solver='lbfgs',
    multi_class='multinomial'
)

nlp_model.fit(X_train_tfidf, y_train_enc)
print('Model training complete.')

## 7. Validation Inference and Logits Export

In [ ]:
y_pred_enc = nlp_model.predict(X_val_tfidf)
logits_val = nlp_model.predict_proba(X_val_tfidf)

binary_f1 = f1_score(y_val_enc, y_pred_enc, average='binary', pos_label=1)
macro_f1  = f1_score(y_val_enc, y_pred_enc, average='macro')

print(f'Validation Binary F1:  {binary_f1:.4f}')
print(f'Validation Macro F1:   {macro_f1:.4f}')
print()
print(classification_report(y_val_enc, y_pred_enc, target_names=le.classes_))

# Save logits and results for evaluation notebook
np.save(os.path.join(OUTPUT_DIR, 'logits_val_nlp_model.npy'), logits_val)

results = {
    'model': 'NLP-TF-IDF-LogisticRegression',
    'binary_f1': binary_f1,
    'macro_f1': macro_f1,
    'classes': list(le.classes_)
}

with open(os.path.join(OUTPUT_DIR, 'nlp_model_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('\nLogits and results saved to models_artifacts/outputs/')

## 8. Summary

In [ ]:
print('=== NLP Model Summary ===')
print(f"Model      : {results['model']}")
print(f"Binary F1  : {results['binary_f1']:.4f}")
print(f"Macro F1   : {results['macro_f1']:.4f}")
print()
print('Outputs written:')
print('  models_artifacts/outputs/logits_val_nlp_model.npy')
print('  models_artifacts/outputs/nlp_model_results.json')